# Create Silver Tables from CSV

This notebook ingests all CSV files from `/Volumes/data_modelling_demo_catalog/source/demo_volume/Silver_Data/` and creates individual Delta tables in `demo_modelling_demo_catalog.silver` schema, with each table named after its source CSV file.

In [0]:
import os

volume_path = "/Volumes/data_modelling_demo_catalog/source/demo_volume/Silver_Data"

# List all CSV files in the volume
csv_files = [f for f in dbutils.fs.ls(volume_path) if f.name.endswith(".csv")]

print(f"Found {len(csv_files)} CSV files:\n")
for f in csv_files:
    print(f"  {f.name}")

In [0]:
target_catalog = "demo_modelling_demo_catalog"
target_schema = "silver"

# Ensure the catalog and schema exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {target_catalog}")
spark.sql(f"USE CATALOG {target_catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema}")
spark.sql(f"USE SCHEMA {target_schema}")

print(f"Catalog '{target_catalog}' and schema '{target_schema}' ready.")

In [0]:
created_tables = []

for f in csv_files:
    file_name = f.name  # e.g. "brands.csv"
    table_name = file_name.replace(".csv", "")  # e.g. "brands"
    full_table_name = f"{target_catalog}.{target_schema}.{table_name}"
    file_path = f.path  # full volume path including trailing /

    print(f"Creating table: {full_table_name} from {file_path}")

    df = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .csv(file_path))

    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(full_table_name))

    row_count = spark.table(full_table_name).count()
    created_tables.append((full_table_name, row_count))
    print(f"  -> {row_count} rows written\n")

print("=" * 60)
print(f"Successfully create d {len(created_tables)} tables:\n")
for table, rows in created_tables:
    print(f"  {table}: {rows} rows")